# LoMa EDisGo-Workshop 27.2.2025

Contents:
1. Topology Setup
2. Worst Case Time Series Creation
3. Grid Investigation
4. Results
5. Additional Time Series


In [ ]:
%load_ext jupyter_black

In [ ]:
import os
import requests
import sys

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

from copy import deepcopy
from numpy.random import default_rng
from pathlib import Path

from edisgo import EDisGo
from edisgo.io.db import engine
from edisgo.tools.logger import setup_logger
from edisgo.flex_opt.battery_storage_operation import apply_reference_operation

In [ ]:
# to make the notebook clearer. not recommendable
import warnings

warnings.filterwarnings("ignore")

## 1 Topology Setup

In this section we load all components into a newly created edisgo object. This includes the lines, buses, transformers, switches, generators, loads, heat pumps and battery storages.

### Standard components

Set up a new edisgo object:

In [ ]:
conf_path = Path.home() / "Downloads" / "egon-data.configuration.yaml"
db_engine = engine(path=conf_path, ssh=True)
ding0_grid = Path.home() / ".edisgo" / "husum_grids" / "35725"

edisgo = EDisGo(ding0_grid=ding0_grid, legacy_ding0_grids=False, engine=db_engine)

The ding0 grids are not up to date and their capacity is not sufficient for the connected loads and generators. To update the imported grids they need to be extended first with the function ```reinforce()```.

Grids are reinforced for their worst case scenarios. The corresponding time series are created with ```set_time_series_worst_case_analysis()```. 

In [ ]:
edisgo.set_time_series_worst_case_analysis()

In [ ]:
edisgo.reinforce()

### Plot grid topology (MV)

The topology can be visualized with the ```plot_mv_grid_topology()```. For ```technologies=True``` the buses sizes and colors are determined to the type and size of the technologies connected to it. 

- red: nodes with substation secondary side
- light blue: nodes distribution substations's primary side
- green: nodes with fluctuating generators
- black: nodes with conventional generators
- grey: disconnecting points
- dark blue: branch trees

In [ ]:
# adjust node sizes to make plot clearer
sizes_dict = {
    "BranchTee": 10000,
    "GeneratorFluctuating": 100000,
    "Generator": 100000,
    "Load": 100000,
    "LVStation": 50000,
    "MVStation": 120000,
    "Storage": 100000,
    "DisconnectingPoint": 75000,
    "else": 200000,
}

sizes_dict = {k: v / 10 for k, v in sizes_dict.items()}

In [ ]:
edisgo.plot_mv_grid_topology(technologies=True, sizes_dict=sizes_dict)

### Topology-Module Data Structure

Let's get familiar with the topology module:

In [ ]:
# generator types
edisgo.topology.generators_df[["p_nom", "type"]].groupby("type").sum()

In [ ]:
# load types
edisgo.topology.loads_df[["p_set", "type"]].groupby("type").sum()

In [ ]:
# load sectors
edisgo.topology.loads_df[["p_set", "sector"]].groupby("sector").sum()

In [ ]:
# amount of lv grids inside the mv grid
len(list(edisgo.topology.mv_grid.lv_grids))

Total number of lines:

In [ ]:
# overall amount of lines
len(edisgo.topology.lines_df)

In [ ]:
# amount of lines in one of the lv grids
len(edisgo.topology.grids[5].lines_df.index)

### Basic components addition and removal

To see how a loaded network can be adapted later on, we add a solar plant to a random bus.

Components can also be added according to their geolocation with the function ```integrate_component_based_on_geolocation()```.

In [ ]:
edisgo.topology.generators_df

Add a generator with the function ```add_component()``` or ```add_generator()```. 

In [ ]:
# determine a random bus
rng = default_rng(1)
rnd_bus = rng.choice(edisgo.topology.buses_df.index, size=1)[0]
generator_type = "solar"

new_generator = edisgo.add_component(
    comp_type="generator", p_nom=0.01, bus=rnd_bus, generator_type=generator_type
)

In [ ]:
edisgo.topology.generators_df

Single components can be removed with ```remove_component()```

In [ ]:
edisgo.remove_component(comp_type="generator", comp_name=new_generator)

In [ ]:
edisgo.topology.generators_df

### Task: 
Add and remobve a 'heat_pump' with the function ```add_component()``` and the function ```remove_component()```.

In [ ]:
edisgo.topology.loads_df

In [ ]:
new_load = edisgo.add_component(
    comp_type="load", p_set=0.01, bus=rnd_bus, type="heat_pump"
)

In [ ]:
edisgo.topology.loads_df

In [ ]:
edisgo.remove_component(comp_type="load", comp_name=new_load)

In [ ]:
edisgo.topology.loads_df

### Add flexible components to grid 

For realistic future grids we also add further components like additional generators, home batteries, (charging points) and heat pumps. The components are added according to the scenario "eGon2035" and the data from the oedb.

In [ ]:
scenario = "eGon2035"

In [ ]:
# copy the edisgo object for later comparisons
edisgo_orig = edisgo.copy()

In [ ]:
# set timeindex to ensure that correct time series for COP and heat pump heat demand are downloaded
timeindex = pd.date_range(f"1/1/{2011} 12:00", periods=4, freq="H")
edisgo.set_timeindex(timeindex=timeindex)

In [ ]:
# Retry if running into "Connection reset by peer" error
# edisgo = deepcopy(edisgo_orig)

edisgo.import_generators(generator_scenario=scenario)
edisgo.import_home_batteries(scenario=scenario)
edisgo.import_heat_pumps(scenario=scenario)

In [ ]:
# This takes too long for the workshop
# edisgo_obj.import_dsm(scenario=scenario)
# edisgo_obj.import_electromobility(
#     data_source="oedb", scenario=scenario
# )

## Task:
Determine the differnet generator types that were installed before and that are installed in the grid now.

In [ ]:
set(edisgo.topology.generators_df["type"])

In [ ]:
set(edisgo_orig.topology.generators_df["type"])

## Task:
Determine the added solar energy power.

In [ ]:
solar_power_new = edisgo.topology.generators_df[
    edisgo.topology.generators_df["type"] == "solar"
]["p_nom"].sum()
solar_power_old = edisgo_orig.topology.generators_df[
    edisgo_orig.topology.generators_df["type"] == "solar"
]["p_nom"].sum()

solar_power_new - solar_power_old

## Task:
Determine the amount of storage units added to the grid with a nominal power (p_nom) larger than 0.01.

In [ ]:
sum(edisgo.topology.storage_units_df["p_nom"] > 0.01)

## Task:
Determine the amount heat pumps connected to the MV level.

In [ ]:
len(
    edisgo.topology.loads_df[
        (edisgo.topology.loads_df["type"] == "heat_pump")
        & (edisgo.topology.loads_df["voltage_level"] == "mv")
    ]
)

## 2 Worst Case Time Series Creation

Create timeseries for the four worst cases MV load case, LV load case, MV feed-in case, LV feed-in case with the function  set_time_series_worst_case_analysis().

In conventional grid expansion planning worst-cases, the heavy load flow and the reverse power flow, are used to determine grid expansion needs. eDisGo allows you to analyze these cases separately or together. Choose between the following options:

* **’feed-in_case’** 
  
  Feed-in and demand for the worst-case scenario "reverse power flow" are generated (e.g. conventional electricity demand is set to 15% of maximum demand for loads connected to the MV grid and 10% for loads connected to the LV grid and feed-in of all generators is set to the nominal power of the generator, except for PV systems where it is by default set to 85% of the nominal power)

  
* **’load_case’**

  Feed-in and demand for the worst-case scenario "heavy load flow" are generated (e.g. demand of all conventional loads is by default set to maximum demand and feed-in of all generators is set to zero)


* **[’feed-in_case’, ’load_case’]**

  Both cases are set up.
  
By default both cases are set up.

Feed-in and demand in the two worst-cases are defined in the [config file 'config_timeseries.cfg'](https://edisgo.readthedocs.io/en/latest/configs.html#config-timeseries) and can be changed by setting different values in the config file. 

In [ ]:
edisgo.set_time_series_worst_case_analysis()

The function creates time series for four time steps since both worst cases are defined seperately for the LV and the MV grid with individual simultanerity factors.

In [ ]:
edisgo.timeseries.timeindex_worst_cases

In [ ]:
# indexing with worst case timeindex
edisgo.timeseries.loads_active_power.loc[
    edisgo.timeseries.timeindex_worst_cases["load_case_mv"]
]

## 3 Grid Investigation

Execute a power flow analysis to determine line overloads and voltage deviations for the MV load case timeseries with the function ```analyze()```:

In [ ]:
# power flow analysis
edisgo.analyze(timesteps=edisgo.timeseries.timeindex_worst_cases["load_case_mv"])

A geoplot with the bus and line colors based on the voltage deviations and line loadings repectively can be created with ```plot_mv_line_loading()```.

In [ ]:
edisgo.plot_mv_line_loading(
    node_color="voltage_deviation",
    timestep=edisgo.timeseries.timeindex_worst_cases["load_case_mv"],
)

For a better overview of the voltage deviations and line loads in the entire grid, edisgo provides histrogram plots.

In [ ]:
edisgo.histogram_voltage(binwidth=0.005)

In [ ]:
edisgo.histogram_relative_line_load(binwidth=0.1)

## 4 Results

In [ ]:
# Reinforce the grid
# mode = "mvlv" for a shorter run time. However, grid reinforcement should generally be conducted in mode="lv" (default)
# since the majority of the reinforcement costs is caused in the lv grid part, especially for high load grids (much EV charging demand and low PV capacity)
edisgo.reinforce(mode="mvlv")

In [ ]:
edisgo.plot_mv_line_loading(
    node_color="voltage_deviation",
    timestep=edisgo.timeseries.timeindex_worst_cases["load_case_mv"],
)

In [ ]:
# power flow analysis to retrieve all bus voltages and line flows
edisgo.analyze(timesteps=edisgo.timeseries.timeindex_worst_cases["load_case_mv"])

In [ ]:
edisgo.histogram_voltage(binwidth=0.005)

In [ ]:
edisgo.histogram_relative_line_load(binwidth=0.1)

The module ```results```holds the outputs of the reinforcement

In [ ]:
# The equipment changes of the reinforcement after the grid setup have to be dropped
edisgo.results.equipment_changes[len(edisgo_orig.results.equipment_changes) :].head()

## Task:
Determine the total costs for the grid reinforcement. The costs for each added component are stored in the data frame ```edisgo.results.grid_expansion_costs```.

In [ ]:
edisgo.results.grid_expansion_costs[len(edisgo_orig.results.grid_expansion_costs) :][
    "total_costs"
].sum()

## 5 Additional Time Series

Besides setting worst case scenarios and the corresponding time series, component time series can also be set with the function ```predefined()```. Either standard profiles for different component types are loaded from a data base or type- (for generators) and sectorwise (for loads) time series can be determined manually and passed to the function. 

The function ```set_time_series_manual()``` can be used to set individual time series for components. 

In [ ]:
# determine interval time series are set for
# timeindex has to be set again to desired time interval because it was overwritten by set_time_series_worst_case()
timeindex = pd.date_range(f"1/1/{2011} 12:00", periods=4, freq="H")
edisgo.set_timeindex(timeindex=timeindex)

In [ ]:
# check which load sectors are included in the Husum grid
set(edisgo.topology.loads_df["sector"])

In [ ]:
# constant load for all time steps for all load types
timeseries_load = pd.DataFrame(
    {
        "industrial": [0.0001] * len(timeindex),
        "cts": [0.0002] * len(timeindex),
        "residential": [0.0002] * len(timeindex),
        "district_heating_resistive_heater": [0.0002] * len(timeindex),
        "individual_heating": [0.0002] * len(timeindex),
    },
    index=timeindex,
)

# annual_consumption of loads is not set in Husum data set
edisgo.topology.loads_df["annual_consumption"] = 700 * edisgo.topology.loads_df["p_set"]

In [ ]:
# check which generator types are included into the grid
set(edisgo.topology.generators_df["type"])

In [ ]:
# constant feed-in for dispatchable generators
timeseries_generation_dispatchable = pd.DataFrame(
    {
        "biomass": [1] * len(timeindex),
        "gas": [1] * len(timeindex),
        "other": [1] * len(timeindex),
    },
    index=timeindex,
)

In [ ]:
# determine fluctuating generators, for which generator-type time series are loaded from a data base
fluctuating_generators = edisgo.topology.generators_df[
    edisgo.topology.generators_df["type"].isin(["solar", "wind"])
].index

In [ ]:
# set active power time series for loads and generators
edisgo.set_time_series_active_power_predefined(
    fluctuating_generators=fluctuating_generators,
    fluctuating_generators_ts="oedb",
    scenario=scenario,
    timeindex=edisgo.timeseries.timeindex,
    conventional_loads_ts=timeseries_load,
    dispatchable_generators_ts=timeseries_generation_dispatchable,
)

In [ ]:
edisgo.timeseries.generators_active_power

In [ ]:
edisgo.timeseries.loads_active_power

In [ ]:
# set heat pump time series 
# set_time_series_active_power_predefined does not consider heat demand
edisgo.apply_heat_pump_operating_strategy()

In [ ]:
edisgo.timeseries.loads_active_power

In [ ]:
# set battery storage time series (not inluded in set_time_series_active_power_predefined())
apply_reference_operation(edisgo)
# returns soe

In [ ]:
edisgo.timeseries.storage_units_active_power

In [ ]:
edisgo.timeseries.generators_reactive_power

In [ ]:
# set reactive power time series
edisgo.set_time_series_reactive_power_control()

In [ ]:
edisgo.timeseries.generators_reactive_power